In [0]:
from pyspark.sql import functions as F
from deltalake import DeltaTable

In [0]:
%run /Workspace/Agmarknet/setup/utilities

In [0]:
print(gold_schema,silver_schema,bronze_schema)

In [0]:
dbutils.widgets.text('catalog','agmarknet')
dbutils.widgets.text('data source','daily prices')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data source')

print(catalog)
print(data_source)

In [0]:
data_path_2025 = f's3://agmarknet-pc/2025/November/*.csv'
data_path_2026 = f's3://agmarknet-pc/2026/*/*.csv'

print(data_path_2025)
print(data_path_2026)

In [0]:
print(f'{catalog}.{bronze_schema}.{data_source}')

###REading 2025 daily data  of months- Oct,Nov,Dec

In [0]:
df1 = (spark.read
       .option("header", "true")
       .option("inferSchema", "false")
       .option("delta.enableChangeDataFeed", "true")
       .option("delimiter", ",")
       .option("multiline", "true")
        .csv(data_path_2025)
        .withColumn('read_timestamp',F.current_timestamp())
        .select("*", "_metadata.file_name")
)

In [0]:
df_m = df1.filter(F.col("Market").contains(","))
display(df_m)

In [0]:
display(df1.limit(20))

In [0]:
df1.printSchema()

In [0]:
df1.count()

In [0]:
df_305 = df1.where(F.col("Commodity_Code")=='305')
display(df_305)

###Reading 2026 daily data  of months- April,May,June

In [0]:
df2 = (spark.read
       .option("header", "true")
       .option("inferSchema", "false")
       .option("delta.enableChangeDataFeed", "true")
       .option("delimiter", ",")
       .option("multiline", "true")
        .csv(data_path_2026)
        .withColumn('read_timestamp',F.current_timestamp())
        .select("*", "_metadata.file_name")
)


In [0]:
df2.count()

In [0]:
df2.printSchema()


In [0]:
df2_305 = df2.where(F.col("Commodity_Code")=='305')
display(df2_305)

In [0]:
df1.count() + df2.count()

In [0]:
### Save the df1 to bronze table
df1.write \
.format("delta") \
.option("delta.enableChangeDataFeed", "true") \
.mode("append") \
.saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

In [0]:
### Save the df2 to bronze table
df2.write \
.format("delta") \
.option("delta.enableChangeDataFeed", "true") \
.mode("append") \
.saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

In [0]:
df_bronze_count = spark.sql(f'select count(*) from {catalog}.{bronze_schema}.{data_source}')
df_bronze_count.show()

###Silver Layer Transformations

In [0]:
df_bronze = spark.sql(f'select * from {catalog}.{bronze_schema}.{data_source}')
df_bronze.show(20)

In [0]:
# cast min price , max price, modal price to double
#cast commodity code to int

df_silver = df_bronze.withColumn("Min_Price", F.col("Min_Price").cast("double")) \
    .withColumn("Max_Price",F.col("Max_Price").cast("double")) \
    .withColumn("Modal_Price",F.col("Modal_Price").cast("double")) \
    .withColumn("Commodity_Code",F.col("Commodity_Code").cast("int"))

In [0]:
df_silver.printSchema()

In [0]:
# analyze date formats before transformation
df_silver_dates = spark.sql(f'select * from {catalog}.{bronze_schema}.{data_source}')

df_formats = (
    df_silver_dates.withColumn(
        "date_format",
        F.when(F.col("Arrival_Date").rlike(r"^\d{2}-\d{2}-\d{4}$"), "dd-MM-yyyy")
         .when(F.col("Arrival_Date").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
         .when(F.col("Arrival_Date").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
         .when(F.col("Arrival_Date").rlike(r"^\d{4}/\d{2}/\d{2}$"), "yyyy/MM/dd")
         .when(F.col("Arrival_Date").rlike(r"^\d{2}-[A-Za-z]{3}-\d{4}$"), "dd-MMM-yyyy")
         .otherwise("Unknown")
    )
)

df_formats.groupBy("date_format").count().show(truncate=False)

In [0]:
df_silver = df_silver.withColumn("Arrival_Date",F.to_date("Arrival_Date",'dd/MM/yyyy'))

In [0]:
df_silver.printSchema()

In [0]:
display(df_silver.limit(10))

In [0]:
### Save the df to silver table
df_silver.write \
.format("delta") \
.option("delta.enableChangeDataFeed", "true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

###Gold layer

In [0]:
silverdf = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")
marketdf = spark.sql(f"select * from {catalog}.{gold_schema}.dim_market")

In [0]:
fact_df = (
    silverdf.alias("f")
    .join(
        marketdf.alias("d"),
        on=["Market", "District", "State"],
        how = "left"
    )
    .select("Arrival_Date"
            ,"Commodity_Code"
            ,"d.Market_Code"
            ,"Min_Price"
            ,"Max_Price"
            ,"Modal_Price"
            )

)

In [0]:
fact_df.show(10)

In [0]:
fact_df.count()

In [0]:
### Save the fact table  to gold schema
fact_df.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed", "true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{gold_schema}.fact_{data_source}')